In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y paddlepaddle paddlepaddle-gpu paddleocr paddlex

In [ ]:
!pip install "paddlepaddle==3.2.2"
!pip install "paddleocr[all]"

In [ ]:
import torch

start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)
start_event.record()

In [ ]:
import cv2
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
from paddleocr import PaddleOCR
import logging
import uuid
import json
import shutil
import time
logging.getLogger("ppocr").setLevel(logging.ERROR)

print("Initializing PaddleOCR...")
try:
    ocr = PaddleOCR(
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        lang='en'
    )
except Exception:
    ocr = PaddleOCR(use_angle_cls=False, lang='en')

print("Warming up the GPU...")
dummy_img = np.zeros((100, 100, 3), dtype=np.uint8)
cv2.imwrite("dummy_warmup.png", dummy_img)
try:
    _ = ocr.predict(input="dummy_warmup.png")
except:
    pass
if os.path.exists("dummy_warmup.png"):
    os.remove("dummy_warmup.png")
print("GPU Warm-up complete.")

def order_points(pts):
    xSorted = pts[np.argsort(pts[:, 0]), :]
    leftMost = xSorted[:2, :]
    rightMost = xSorted[2:, :]
    leftMost = leftMost[np.argsort(leftMost[:, 1]), :]
    (tl, bl) = leftMost
    D = np.linalg.norm(rightMost - tl, axis=1)
    (br, tr) = rightMost[np.argsort(D)[::-1], :]
    return np.array([tl, tr, br, bl], dtype="float32")

def get_distance_to_line(p, p1, p2):
    num = np.abs((p2[1] - p1[1])*p[0] - (p2[0] - p1[0])*p[1] + p2[0]*p1[1] - p2[1]*p1[0])
    den = np.linalg.norm(p2 - p1)
    if den == 0: return float('inf')
    return num / den

def get_projection_t(p, p1, p2):
    v = p2 - p1
    w = p - p1
    den = np.dot(v, v)
    if den == 0: return -1
    return np.dot(w, v) / den

def intersect_lines(line1, line2):
    vx1, vy1, x1, y1 = line1
    vx2, vy2, x2, y2 = line2
    D = vy1 * vx2 - vx1 * vy2
    if D == 0: return None
    t1 = (vx2 * (y2 - y1) - vy2 * (x2 - x1)) / D
    return np.array([x1 + t1 * vx1, y1 + t1 * vy1], dtype="float32")

def fix_orientation_by_symbol_count(image, ocr_instance):
    def count_symbols_using_predict(img_data):
        run_id = uuid.uuid4().hex
        temp_img_name = f"temp_img_{run_id}.png"
        temp_output_dir = f"temp_output_{run_id}"

        cv2.imwrite(temp_img_name, img_data)
        symbol_count = 0
        try:
            result = ocr_instance.predict(input=temp_img_name)
            os.makedirs(temp_output_dir, exist_ok=True)
            for res in result:
                res.save_to_json(temp_output_dir)

            json_files = glob.glob(os.path.join(temp_output_dir, "*.json"))
            for jf in json_files:
                with open(jf, 'r') as f:
                    data = json.load(f)
                    if "rec_texts" in data and isinstance(data["rec_texts"], list):
                        text_list = data["rec_texts"]
                        for text in text_list:
                            if text:
                                clean_text = str(text).strip()
                                clean_text = clean_text.replace(".", "")
                                symbol_count += len(clean_text)
        except Exception:
            pass
        finally:
            if os.path.exists(temp_img_name):
                os.remove(temp_img_name)
            if os.path.exists(temp_output_dir):
                shutil.rmtree(temp_output_dir)

        return symbol_count

    count_normal = count_symbols_using_predict(image)
    rotated_img = cv2.rotate(image, cv2.ROTATE_180)
    count_rotated = count_symbols_using_predict(rotated_img)

    if count_normal > 0 or count_rotated > 0:
        print(f"  -> OCR Counts | Normal: {count_normal} | Rotated: {count_rotated}")

    if count_rotated > count_normal:
        print("  -> Rotated version has more text. Flipping.")
        return cv2.rotate(image, cv2.ROTATE_180)

    return image

def process_image(img_path, output_path, visualize=False):
    try:
        img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        if img is None:
            return

        if img.shape[2] == 4:
            base_bgr = img[:, :, :3]
        else:
            base_bgr = img

        gray = cv2.cvtColor(base_bgr, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            return

        c = max(contours, key=cv2.contourArea)
        pts = c.reshape(-1, 2)

        rect = cv2.minAreaRect(c)
        box = cv2.boxPoints(rect)
        box = order_points(box)
        tl, tr, br, bl = box
        edges = [(tl, tr), (tr, br), (br, bl), (bl, tl)]

        edge_clusters = {0: [], 1: [], 2: [], 3: []}
        for p in pts:
            min_dist = float('inf')
            best_edge = -1
            best_t = -1
            for i, (p1, p2) in enumerate(edges):
                dist = get_distance_to_line(p, p1, p2)
                if dist < min_dist:
                    min_dist = dist
                    best_edge = i
                    best_t = get_projection_t(p, p1, p2)

            if 0.05 < best_t < 0.95:
                edge_clusters[best_edge].append(p)

        lines = []
        for i in range(4):
            cluster = np.array(edge_clusters[i], dtype=np.float32)
            if len(cluster) > 10:
                max_inliers = 0
                best_inlier_mask = None
                for _ in range(200):
                    idx = np.random.choice(len(cluster), 2, replace=False)
                    pt1, pt2 = cluster[idx[0]], cluster[idx[1]]
                    v = pt2 - pt1
                    v_norm = np.linalg.norm(v)
                    if v_norm < 1e-5: continue
                    n_vec = np.array([-v[1], v[0]]) / v_norm
                    dists = np.abs(np.dot(cluster - pt1, n_vec))
                    inliers = dists < 2.0
                    num_inliers = np.sum(inliers)
                    if num_inliers > max_inliers:
                        max_inliers = num_inliers
                        best_inlier_mask = inliers

                if best_inlier_mask is not None and max_inliers > 2:
                    clean_cluster = cluster[best_inlier_mask]
                    [vx, vy, x, y] = cv2.fitLine(clean_cluster, cv2.DIST_L2, 0, 0.01, 0.01)
                else:
                    [vx, vy, x, y] = cv2.fitLine(cluster, cv2.DIST_L2, 0, 0.01, 0.01)
            else:
                p1, p2 = edges[i]
                vx = p2[0] - p1[0]
                vy = p2[1] - p1[1]
                norm = np.sqrt(vx**2 + vy**2)
                if norm == 0: norm = 1
                vx, vy, x, y = vx/norm, vy/norm, p1[0], p1[1]
            lines.append((vx, vy, x, y))

        tl_virtual = intersect_lines(lines[0], lines[3])
        tr_virtual = intersect_lines(lines[0], lines[1])
        br_virtual = intersect_lines(lines[2], lines[1])
        bl_virtual = intersect_lines(lines[2], lines[3])
        virtual_corners = np.array([tl_virtual, tr_virtual, br_virtual, bl_virtual], dtype="float32")

        widthA = np.sqrt(((br_virtual[0] - bl_virtual[0]) ** 2) + ((br_virtual[1] - bl_virtual[1]) ** 2))
        widthB = np.sqrt(((tr_virtual[0] - tl_virtual[0]) ** 2) + ((tr_virtual[1] - tl_virtual[1]) ** 2))
        maxWidth = max(int(np.array(widthA).item()), int(np.array(widthB).item()))

        heightA = np.sqrt(((tr_virtual[0] - br_virtual[0]) ** 2) + ((tr_virtual[1] - br_virtual[1]) ** 2))
        heightB = np.sqrt(((tl_virtual[0] - bl_virtual[0]) ** 2) + ((tl_virtual[1] - bl_virtual[1]) ** 2))
        maxHeight = max(int(np.array(heightA).item()), int(np.array(heightB).item()))

        if maxHeight > maxWidth:
            maxWidth, maxHeight = maxHeight, maxWidth
            virtual_corners = np.roll(virtual_corners, 1, axis=0)

        dst = np.float32([
            [0, 0],
            [maxWidth - 1, 0],
            [maxWidth - 1, maxHeight - 1],
            [0, maxHeight - 1]
        ])

        b, g, r = cv2.split(base_bgr)
        img_bgra = cv2.merge((b, g, r, thresh))
        M = cv2.getPerspectiveTransform(virtual_corners, dst)
        warped_transparent = cv2.warpPerspective(img_bgra, M, (maxWidth, maxHeight), borderValue=(0, 0, 0, 0))

        if maxWidth > 50 and maxHeight > 20:
             warped_transparent = fix_orientation_by_symbol_count(warped_transparent, ocr)

        cv2.imwrite(output_path, warped_transparent)
        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error processing {os.path.basename(img_path)}: {str(e)}")


IMAGES_DIR = '/content/drive/MyDrive/nonCropPictures/_boxed_outputs/cutouts_png'
OUTPUT_DIR = "/content/drive/MyDrive/allRotatedPhotos"

os.makedirs(OUTPUT_DIR, exist_ok=True)
image_paths = glob.glob(os.path.join(IMAGES_DIR, "*.png"))

total_pipeline_time = 0.0

if not image_paths:
    print(f"No PNG images found in {IMAGES_DIR}.")
else:
    print(f"Found {len(image_paths)} images to process. Starting timer...")

    for idx, img_path in enumerate(image_paths, 1):
        filename = os.path.basename(img_path)
        output_path = os.path.join(OUTPUT_DIR, filename)

        start_time = time.time()

        process_image(img_path, output_path, visualize=False)

        end_time = time.time()

        image_time = end_time - start_time
        total_pipeline_time += image_time
        print(f"⏱️ Rectification & OCR Time for {filename}: {image_time:.2f} seconds\n")

    print("========================================")
    print("Batch processing complete!")
    print(f"Total Computation Time: {total_pipeline_time:.2f} seconds")
    print(f"Average Time per Image: {(total_pipeline_time / len(image_paths)):.2f} seconds")
    print("========================================")